# 02 — TurboQuant : Compression du KV Cache

**Objectif** : Implémenter TurboQuant (PolarQuant + QJL) pour compresser le KV cache de Qwen3.5-2B fine-tuné.

Pipeline :
```
fine-tune ✅ → TurboQuant compression (ce notebook) → évaluation before/after
```

**Référence** : [TurboQuant (arXiv:2504.19874)](https://arxiv.org/abs/2504.19874) — Google, mars 2026

---
## Architecture TurboQuant

```
Vecteur KV x ∈ Rᵈ (float16, d=128)
        │
        ▼  Rotation aléatoire Π (orthogonale)
   y = Π·x  ~  N(0, 1/d)  par coordonnée
        │
        ▼  Lloyd-Max (b-1 bits) — codebook précalculé
   idx  +  x̃_mse
        │
        ▼  Résidu r = x - x̃_mse
   sign(S·r)  →  QJL 1-bit  →  correction non-biaisée
        │
        ▼
   x̃ = x̃_mse + correction_QJL   (unbiaisé, ~3 bits/coord)
```

**Résultat attendu** : compression ×5 du KV cache, qualité préservée.

## 1. Setup & Drive

In [ ]:
!pip install unsloth datasets transformers accelerate -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/efficient-llm-pipeline'
LORA_PATH  = f'{DRIVE_BASE}/models/qwen-gsm8k-lora'

assert os.path.exists(LORA_PATH), f'Adaptateurs LoRA introuvables : {LORA_PATH}'
print(f'Adaptateurs trouvés : {os.listdir(LORA_PATH)}')

## 2. Chargement du modèle fine-tuné

On charge directement depuis le dossier Drive qui contient les adaptateurs LoRA.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

vram = torch.cuda.memory_allocated() / 1e9
print(f'Modèle chargé | VRAM : {vram:.2f} GB')

## 3. Lloyd-Max codebook

Le codebook optimal pour `N(0,1)` est précalculé une fois.
On résout le problème k-means continu 1D par algorithme de Lloyd itératif.

Pour `b` bits → `2^b` niveaux de quantification.

In [ ]:
import numpy as np

def lloyd_max_gaussian(b: int, n_samples: int = 100_000) -> np.ndarray:
    """
    Calcule les centroïdes Lloyd-Max optimaux pour N(0,1).
    On normalise ensuite par la std attendue (1/sqrt(d)) à l'usage.

    Args:
        b: nombre de bits (2^b niveaux)
        n_samples: échantillons pour l'estimation
    Returns:
        centroids: array de shape (2^b,), trié
    """
    n_levels = 2 ** b
    samples  = np.random.randn(n_samples)

    # Initialisation uniforme entre les percentiles 1% et 99%
    centroids = np.linspace(
        np.percentile(samples, 1),
        np.percentile(samples, 99),
        n_levels
    )

    for _ in range(200):
        dists    = np.abs(samples[:, None] - centroids[None, :])  # (n_samples, n_levels)
        assigned = np.argmin(dists, axis=1)

        new_centroids = np.array([
            samples[assigned == k].mean() if (assigned == k).any() else centroids[k]
            for k in range(n_levels)
        ])

        if np.max(np.abs(new_centroids - centroids)) < 1e-6:
            break
        centroids = new_centroids

    return np.sort(centroids)


# Précalcul des codebooks pour b=1,2,3,4
CODEBOOKS = {b: lloyd_max_gaussian(b) for b in range(1, 5)}

print('Codebooks Lloyd-Max précalculés :')
for b, cb in CODEBOOKS.items():
    print(f'  b={b} ({2**b} niveaux) : {cb.round(3)}')

## 4. TurboQuant — implémentation complète

Deux classes :
- `TurboQuantMSE` : rotation aléatoire + Lloyd-Max → minimise l'erreur quadratique
- `TurboQuantProd` : TurboQuantMSE(b-1 bits) + QJL 1-bit sur le résidu → produits scalaires non-biaisés

In [ ]:
import math
import torch
import torch.nn as nn


class TurboQuantMSE(nn.Module):
    """
    TurboQuant MSE-optimal : rotation aléatoire + quantification Lloyd-Max.

    Pour x ∈ R^d :
        Quant  : idx = argmin_k |Π·x - c_k|  (b bits/coord)
        DeQuant: x̃  = Πᵀ · {c_{idx_j}}
    """

    def __init__(self, dim: int, bits: int, seed: int = 42):
        super().__init__()
        self.dim  = dim
        self.bits = bits

        # Matrice de rotation orthogonale aléatoire (fixe, non entraînable)
        torch.manual_seed(seed)
        G = torch.randn(dim, dim)
        Q, _ = torch.linalg.qr(G)   # Q·Qᵀ = I
        self.register_buffer('rotation', Q)

        # Codebook Lloyd-Max
        cb = torch.tensor(CODEBOOKS[bits], dtype=torch.float32)
        self.register_buffer('codebook', cb)

        # Écart-type attendu après rotation : 1/sqrt(d)
        self.scale = 1.0 / (dim ** 0.5)

    def quantize(self, x: torch.Tensor):
        """x : (..., dim)  →  (idx, x_dequant)"""
        shape  = x.shape
        x_flat = x.reshape(-1, self.dim).float()   # (N, d)

        # Rotation : y = Π·x  ~  N(0, 1/d) par coord
        y      = x_flat @ self.rotation.T          # (N, d)
        y_norm = y / self.scale                    # normalise vers N(0,1)

        # Quantification : centroïde le plus proche
        dists = (y_norm.unsqueeze(-1) - self.codebook).abs()  # (N, d, 2^b)
        idx   = dists.argmin(dim=-1)                           # (N, d)

        # Déquantification dans l'espace rotaté puis rotation inverse
        y_hat = self.codebook[idx] * self.scale    # (N, d)
        x_hat = y_hat @ self.rotation              # (N, d)  — Πᵀ·y_hat

        return idx.reshape(shape), x_hat.reshape(shape)

    def forward(self, x):
        _, x_hat = self.quantize(x)
        return x_hat


class TurboQuantProd(nn.Module):
    """
    TurboQuant inner-product unbiaisé : MSE(b-1 bits) + QJL 1-bit sur le résidu.

    Garantit : E[<y, x̃>] = <y, x>  pour tout y.
    """

    def __init__(self, dim: int, bits: int, seed: int = 42):
        super().__init__()
        assert bits >= 2, 'bits >= 2 requis (b-1 pour MSE + 1 pour QJL)'
        self.dim  = dim
        self.bits = bits

        # Quantificateur MSE sur b-1 bits
        self.mse = TurboQuantMSE(dim, bits - 1, seed=seed)

        # Matrice QJL : S ∈ R^(d×d), entrées ~ N(0,1)
        torch.manual_seed(seed + 1)
        self.register_buffer('S', torch.randn(dim, dim))

        # Facteur de correction non-biaisé : sqrt(π/2) / d
        self.qjl_scale = math.sqrt(math.pi / 2) / dim

    def quantize(self, x: torch.Tensor):
        """x : (..., dim)  →  (idx, qjl_bits, residual_norm, x_hat)"""
        shape  = x.shape
        x_flat = x.reshape(-1, self.dim).float()   # (N, d)

        # Étape 1 : MSE sur b-1 bits
        idx, x_mse = self.mse.quantize(x_flat)    # idx: (N,d)  x_mse: (N,d)

        # Étape 2 : résidu
        r      = x_flat - x_mse                   # (N, d)
        r_norm = r.norm(dim=-1, keepdim=True)      # (N, 1)

        # Étape 3 : QJL — signe de la projection
        r_proj   = r @ self.S.T                   # (N, d)
        qjl_bits = r_proj.sign().to(torch.int8)   # (N, d) ∈ {-1, +1}

        # Reconstruction : MSE + correction QJL non-biaisée
        correction = self.qjl_scale * r_norm * (qjl_bits.float() @ self.S)
        x_hat      = x_mse + correction           # (N, d)

        return (
            idx.reshape(shape),
            qjl_bits.reshape(shape),
            r_norm.reshape(*shape[:-1], 1),
            x_hat.reshape(shape),
        )

    def dequantize(self, idx, qjl_bits, r_norm):
        """Reconstruction depuis les données compressées."""
        shape  = idx.shape
        idx_f  = idx.reshape(-1, self.dim)
        qjl_f  = qjl_bits.reshape(-1, self.dim).float()
        norm_f = r_norm.reshape(-1, 1)

        y_hat  = self.mse.codebook[idx_f] * self.mse.scale
        x_mse  = y_hat @ self.mse.rotation
        correction = self.qjl_scale * norm_f * (qjl_f @ self.S)

        return (x_mse + correction).reshape(shape)

    def forward(self, x):
        _, _, _, x_hat = self.quantize(x)
        return x_hat


print('TurboQuantMSE et TurboQuantProd définis.')

## 5. Validation mathématique

Avant d'attacher TurboQuant au modèle, on vérifie les deux propriétés théoriques :
1. **MSE** : l'erreur de reconstruction diminue avec b
2. **Unbiaisé** : `E[<y, x̃>] ≈ <y, x>` pour TurboQuantProd

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
DIM    = 128   # dimension typique d'une tête d'attention Qwen3.5-2B
N      = 1000  # vecteurs test

# Vecteurs normalisés sur la sphère unité (comme dans le paper)
x = torch.randn(N, DIM, device=device)
x = x / x.norm(dim=-1, keepdim=True)
y = torch.randn(N, DIM, device=device)

print('=== TurboQuantMSE — erreur de reconstruction ===')
for b in [1, 2, 3, 4]:
    quant = TurboQuantMSE(DIM, b).to(device)
    x_hat = quant(x)
    mse   = (x - x_hat).pow(2).mean().item()
    print(f'  b={b} bits → MSE = {mse:.4f}')

print()
print('=== TurboQuantProd — biais sur produit scalaire ===')
true_dot = (y * x).sum(dim=-1)

for b in [2, 3, 4]:
    quant = TurboQuantProd(DIM, b).to(device)
    _, _, _, x_hat = quant.quantize(x)
    approx_dot = (y * x_hat).sum(dim=-1)

    bias = (approx_dot - true_dot).mean().item()
    rmse = (approx_dot - true_dot).pow(2).mean().sqrt().item()
    print(f'  b={b} bits → biais = {bias:.4f} (attendu ≈ 0) | RMSE = {rmse:.4f}')

## 6. TurboQuantCache — hook KV cache

On remplace le `DynamicCache` de HuggingFace par notre version compressée.
À chaque token généré, K et V sont compressés avant stockage et décompressés à la lecture.

```
Attention layer
    ├── compute K, V  (float16)
    │       ▼  TurboQuantCache.update()
    │   compress → stocker (idx, qjl_bits, r_norm)
    │       ▼  retourner K̃, Ṽ décompressés
    └── softmax(Q·K̃ᵀ/√d) · Ṽ
```

In [ ]:
from transformers import DynamicCache


class TurboQuantCache(DynamicCache):
    """
    Cache KV compressé avec TurboQuant.
    Remplace DynamicCache — interface identique, stockage compressé.
    """

    def __init__(self, dim: int, bits: int = 3):
        super().__init__()
        self.quantizer     = TurboQuantProd(dim, bits)
        self._comp_k       = []   # (idx, qjl, norm) par couche
        self._comp_v       = []
        self.bits_original = 0
        self.bits          = bits

    def update(self, key_states, value_states, layer_idx, cache_kwargs=None):
        device = key_states.device
        self.quantizer = self.quantizer.to(device)

        # Compresser K et V
        k_idx, k_qjl, k_norm, k_hat = self.quantizer.quantize(key_states)
        v_idx, v_qjl, v_norm, v_hat = self.quantizer.quantize(value_states)

        if layer_idx >= len(self._comp_k):
            self._comp_k.append((k_idx, k_qjl, k_norm))
            self._comp_v.append((v_idx, v_qjl, v_norm))
        else:
            ok = self._comp_k[layer_idx]
            ov = self._comp_v[layer_idx]
            self._comp_k[layer_idx] = (
                torch.cat([ok[0], k_idx],  dim=2),
                torch.cat([ok[1], k_qjl],  dim=2),
                torch.cat([ok[2], k_norm],  dim=2),
            )
            self._comp_v[layer_idx] = (
                torch.cat([ov[0], v_idx],  dim=2),
                torch.cat([ov[1], v_qjl],  dim=2),
                torch.cat([ov[2], v_norm],  dim=2),
            )

        self.bits_original += (key_states.numel() + value_states.numel()) * 16

        # Retourner les versions décompressées pour l'attention
        return k_hat.to(key_states.dtype), v_hat.to(value_states.dtype)

    @property
    def compression_ratio(self):
        return 16 / self.bits

    @property
    def memory_saved_mb(self):
        saved_bits = self.bits_original * (1 - self.bits / 16)
        return saved_bits / 8 / 1e6


print('TurboQuantCache défini.')

## 7. Benchmark — VRAM et latence

On compare baseline vs TurboQuant sur 10 questions GSM8K.
Métriques : VRAM peak, temps de génération, accuracy.

In [ ]:
import time
from datasets import load_dataset

dataset   = load_dataset('openai/gsm8k', 'main', split='test')
QUESTIONS = [dataset[i]['question'] for i in range(10)]
EXPECTED  = [dataset[i]['answer'].split('####')[-1].strip() for i in range(10)]

SYSTEM_PROMPT = (
    'Tu es un assistant mathématique expert. '
    'Décompose chaque problème étape par étape en montrant tous les calculs. '
    "Termine toujours par '#### <réponse>' sur la dernière ligne."
)

HEAD_DIM = model.config.hidden_size // model.config.num_attention_heads
print(f'Head dim : {HEAD_DIM}')

def generate(question, use_turboquant=False, max_new_tokens=256):
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT}]},
        {'role': 'user',   'content': [{'type': 'text', 'text': question}]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')

    kwargs = {'input_ids': inputs, 'max_new_tokens': max_new_tokens, 'do_sample': False}
    if use_turboquant:
        kwargs['past_key_values'] = TurboQuantCache(dim=HEAD_DIM, bits=3)

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**kwargs)
    elapsed = time.time() - t0
    vram    = torch.cuda.max_memory_reserved() / 1e9

    response = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    return response, elapsed, vram

In [ ]:
results = {'baseline': [], 'turboquant': []}

print('Benchmark sur 10 questions GSM8K...\n')
print(f'{"Q":<4} {"Baseline":>20} {"TurboQuant":>20}')
print(f'{"":4} {"time  vram  acc":>20} {"time  vram  acc":>20}')
print('-' * 50)

for i, (q, expected) in enumerate(zip(QUESTIONS, EXPECTED)):
    resp_b, t_b, vram_b = generate(q, use_turboquant=False)
    resp_t, t_t, vram_t = generate(q, use_turboquant=True)

    ans_b = resp_b.split('####')[-1].strip() if '####' in resp_b else '?'
    ans_t = resp_t.split('####')[-1].strip() if '####' in resp_t else '?'

    results['baseline'].append({'time': t_b, 'vram': vram_b, 'correct': ans_b == expected})
    results['turboquant'].append({'time': t_t, 'vram': vram_t, 'correct': ans_t == expected})

    print(f'Q{i+1:<3} {t_b:.1f}s {vram_b:.2f}GB {"✓" if ans_b==expected else "✗":>3}   '
          f'{t_t:.1f}s {vram_t:.2f}GB {"✓" if ans_t==expected else "✗":>3}')

# Résumé
b_acc = sum(r['correct'] for r in results['baseline'])  / 10
t_acc = sum(r['correct'] for r in results['turboquant']) / 10
b_time = sum(r['time'] for r in results['baseline'])  / 10
t_time = sum(r['time'] for r in results['turboquant']) / 10
b_vram = sum(r['vram'] for r in results['baseline'])  / 10
t_vram = sum(r['vram'] for r in results['turboquant']) / 10

print(f"""
{'='*50}
RÉSULTATS (10 questions GSM8K)
{'='*50}
                  Baseline    TurboQuant 3-bit
Accuracy        : {b_acc*100:.0f}%          {t_acc*100:.0f}%
Temps moyen     : {b_time:.1f}s          {t_time:.1f}s
VRAM peak moy.  : {b_vram:.2f} GB       {t_vram:.2f} GB
Compression     : 1×             {16/3:.1f}× (16→3 bits)
""")

## 8. Sauvegarde de la configuration TurboQuant

On sauvegarde les matrices (rotation + QJL) et les codebooks pour `03_evaluate.ipynb`.

In [ ]:
TURBOQUANT_PATH = f'{DRIVE_BASE}/models/turboquant_config'
os.makedirs(TURBOQUANT_PATH, exist_ok=True)

tq = TurboQuantProd(dim=HEAD_DIM, bits=3).cuda()

torch.save({
    'rotation' : tq.mse.rotation.cpu(),
    'S'        : tq.S.cpu(),
    'codebooks': {str(b): torch.tensor(cb) for b, cb in CODEBOOKS.items()},
    'config'   : {'dim': HEAD_DIM, 'bits': 3, 'model': 'Qwen/Qwen3.5-2B'},
    'benchmark': results,
}, f'{TURBOQUANT_PATH}/turboquant.pt')

print(f'Configuration sauvegardée → {TURBOQUANT_PATH}/turboquant.pt')

## Récapitulatif

| Composant | Rôle | Bits |
|---|---|---|
| Rotation Π | Gaussianiser les coordonnées | — |
| Lloyd-Max | Quantification MSE-optimale | b-1 |
| QJL | Correction résidu, non-biaisé | 1 |
| **Total** | **Inner-product unbiaisé** | **b** |

**Suite → `03_evaluate.ipynb`** : évaluation complète before/after sur GSM8K (accuracy, latence, VRAM).

---
*[efficient-llm-pipeline](https://github.com/YanissAmz/efficient-llm-pipeline)*